In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.stats import norm, chi2

class PortfolioRiskEngine:
    def __init__(self, tickers: list, weights: list, portfolio_val: float, period: str = '3y', confidence_level: float = 0.99):
        self.tickers = tickers
        # create portfolio with a numpy array (needed to do matrix algebra later)
        self.weights = np.array(weights)
        self.portfolio_val = portfolio_val
        self.period = period
        self.confidence_level = confidence_level

        # internal attributes populated during execution
        self.log_returns = None
        self.cov_matrix = None
        self.port_pnl = None

        # run data ingestion pipeline automatically on instantiation
        self._fetch_and_process_data()

    def _fetch_and_process_data(self):
        """Phase 1: Importing Data and Creating Math Framework"""
        # pulling 3 years of data
        data = yf.download(self.tickers, period=self.period, auto_adjust=True, progress=False)['Close']

        # log returns are needed for later calculations (contains dataframe for daily log returns)
        self.log_returns = np.log(data / data.shift(1)).dropna()

        # calculate the sample covariance matrix of the log returns (4x4 matrix)
        self.cov_matrix = self.log_returns.cov()

        # dot product of daily log returns and weights (approximation since daily returns are small)
        historical_portfolio_returns = self.log_returns.dot(self.weights)

        # convert percentage returns into actual dollar PnL across our total portfolio value
        self.port_pnl = historical_portfolio_returns * self.portfolio_val

    def calculate_parametric_var(self, horizon_days: int = 1) -> dict:
        """Phase 1: Parametric VaR using variance-covariance matrix."""
        # portfolio variance = w^T * Sigma * w (quadratic form multiplying row weights, matrix, and column weights to get scalar variance)
        port_variance = np.dot(self.weights.T, np.dot(self.cov_matrix, self.weights))
        port_std_dev = np.sqrt(port_variance)

        # z-score for confidence level (inverse cdf of normal distribution)
        z_score = norm.ppf(self.confidence_level)

        # 1 day value at risk in percent terms (mean set to 0 because daily drift is negligible)
        var_1d_pct = z_score * port_std_dev
        var_1d_usd = var_1d_pct * self.portfolio_val

        # scale up using square root of time rule, assuming iid returns
        var_horizon_usd = var_1d_usd * np.sqrt(horizon_days)

        return {
            "method": "Parametric (Normal)",
            "daily_volatility_pct": float(round(port_std_dev * 100, 2)),
            f"{horizon_days}_day_var_usd": float(round(var_horizon_usd, 2))
        }

    def calculate_historical_var(self, horizon_days: int = 1) -> dict:
        """Phase 2: Historical Simulation VaR."""
        # dynamic percentile calculation (100 - 99 = 1st percentile to grab left-tail loss cutoff)
        percentile = (1.0 - self.confidence_level) * 100

        # extract exact dollar threshold from historical data and take absolute value to report as positive risk
        hist_var_1d_usd = np.abs(np.percentile(self.port_pnl, percentile))

        # scale 1-day historical VaR to multi-day using the square root of time rule
        hist_var_horizon_usd = hist_var_1d_usd * np.sqrt(horizon_days)

        return {
            "method": "Historical Simulation",
            f"{horizon_days}_day_var_usd": float(round(hist_var_horizon_usd, 2))
        }

    def run_kupiec_backtest(self) -> dict:
        """Phase 3: Kupiec POF Backtest for the Historical VaR model."""
        # get the absolute 1-day historical VaR threshold
        var_result = self.calculate_historical_var(horizon_days=1)
        var_threshold = -var_result["1_day_var_usd"] # make it negative to compare against PnL

        # count how many days our actual historical PnL was worse than the VaR threshold
        breaches = self.port_pnl < var_threshold
        x = breaches.sum()
        n = len(self.port_pnl)
        p = 1.0 - self.confidence_level

        # Likelihood Ratio test statistic
        hat_p = x / n
        if x == 0:
            lr_stat = -2 * np.log((1 - p)**n)
        elif x == n:
            lr_stat = -2 * np.log(p**n)
        else:
            lr_stat = -2 * np.log(
                (((1 - p)**(n - x)) * (p**x)) / (((1 - hat_p)**(n - x)) * (hat_p**x))
            )

        # check if p-value is > 0.05 to accept the model
        p_value = 1 - chi2.cdf(lr_stat, df=1)
        passed = p_value > 0.05

        return {
            "total_observations": int(n),
            "breaches": int(x),
            "expected_breaches": float(round(n * p, 2)),
            "p_value": float(round(p_value, 4)),
            "model_accepted": bool(passed)
        }

# --- Execution Example ---
if __name__ == "__main__":
    engine = PortfolioRiskEngine(
        tickers=['AAPL', 'MSFT', 'GLD', 'TLT'],
        weights=[0.25, 0.25, 0.25, 0.25],
        portfolio_val=1_000_000
    )

    print("Parametric VaR:", engine.calculate_parametric_var(horizon_days=10))
    print("Historical VaR:", engine.calculate_historical_var(horizon_days=10))
    print("Kupiec Backtest:", engine.run_kupiec_backtest())

Parametric VaR: {'method': 'Parametric (Normal)', 'daily_volatility_pct': 0.83, '10_day_var_usd': 60962.05}
Historical VaR: {'method': 'Historical Simulation', '10_day_var_usd': 70778.86}
Kupiec Backtest: {'total_observations': 752, 'breaches': 8, 'expected_breaches': 7.52, 'p_value': 0.8618, 'model_accepted': True}
